In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_14.pickle

trying: ['file_name_2017', 'file_name_2018']
me:  8
me:  8
trying: ['file_name_2017', 'file_name_2018']
me:  8
me:  8
trying: ['count_then_return_percent_17']
me:  16
trying: ['bar_chart_multiple_choice_17']
me:  16
trying: ['orientation_for_chart']
me:  16
trying: ['country_df_combined']
me:  18
trying: ['subset_of_countries']
me:  18
trying: ['add_year_column_to_dataframes_18']
me:  18
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  18
trying: ['question_name']
me:  18
trying: ['bar_chart_multiple_years_18']
me:  18
trying: ['question_name_alternate']
me:  18
trying: ['count_then_return_percent_18']
me:  18
trying: ['responses_in_order']
me:  24
trying: ['count_then_return_percent_21']
me:  24
trying: ['bar_chart_multiple_choice_21']
me:  24
trying: ['title_for_y_axis']
me:  28
trying: ['title_for_x_axis']
me:  18
trying: ['base_dir_2020']
me:  8
trying: ['base_dir_2017']
me:  8
trying: ['bar_chart_multiple_years_22']
me:  26
trying: ['directory']
me:  8
trying: ['age_

me:  26
trying: ['file_name_2022']
me:  8
trying: ['base_dir_2021']
me:  8
trying: ['add_year_column_to_dataframes_22']
me:  26
trying: ['column_of_interest']
me:  26
trying: ['question_of_interest']
me:  28
trying: ['count_then_return_percent_22']
me:  26
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  26
trying: ['percentages']
me:  28
trying: ['count_then_return_percent_23']
me:  28
trying: ['pd']
me:  0
trying: ['title_for_chart']
me:  28
trying: ['px']
me:  0
trying: ['create_choropleth_map']
me:  2
trying: ['bar_chart_multiple_choice_multiple_selection']
me:  2
trying: ['bar_chart_multiple_choice']
me:  2
trying: ['bar_chart_multiple_years']
me:  2
trying: ['count_then_return_percent']
me:  4
trying: ['add_year_column_to_dataframes']
me:  4
trying: ['glob']
me:  0
trying: ['bar_chart_multiple_choice_23']
me:  28
trying: ['grab_subset_of_data']
me:  4
trying: ['combine_subset_of_data_from_multiple_years']
me:  6
trying: ['load_survey_data']
me:  6
trying: ['file_nam

me:  18
trying: ['add_year_column_to_dataframes_20']
me:  22
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  22
trying: ['count_then_return_percent_20']
me:  22
trying: ['bar_chart_multiple_years_20']
me:  22
trying: ['responses_df_2020']


me:  8
trying: ['responses_df_2021']


me:  8
trying: ['orig_output']
me:  15
trying: ['create_dataframe_of_counts_16']
me:  14
trying: ['label_for_legend']
me:  14
trying: ['responses_per_country_df']
me:  14
trying: ['create_choropleth_map_16']
me:  14
trying: ['percentages_per_country_df']
me:  14
trying: ['responses_df_2018']


me:  10
trying: ['replace_hyphen_with_en_dash']
me:  10
trying: ['responses_df_2019']


me:  10
trying: ['bar_chart_multiple_choice_19']
me:  20
trying: ['count_then_return_percent_19']
me:  20
Declaring variable pd
Declaring variable px
Declaring variable glob
Declaring variable sns
Declaring variable go
Declaring variable warnings
Declaring variable np
Declaring variable create_choropleth_map
Declaring variable bar_chart_multiple_choice_multiple_selection
Declaring variable bar_chart_multiple_choice
Declaring variable bar_chart_multiple_years
Declaring variable count_then_return_percent
Declaring variable add_year_column_to_dataframes
Declaring variable grab_subset_of_data
Declaring variable convert_df_of_counts_to_percentages
Declaring variable create_dataframe_of_counts
Declaring variable combine_subset_of_data_from_multiple_years
Declaring variable load_survey_data
Declaring variable combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions
Declaring variable file_name_2017
Declaring variable file_name_2018
Declaring variable file_nam

In [4]:
%%RecordEvent
%%time
### cell 15 ###

### cell 15 (optimized for cudf)

def bar_chart_multiple_choice_24(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    # response_counts is already a cudf.Series
    response_counts.index.name = ''
    # write top choices directly from the cudf.Series without converting back to pandas
    response_counts.iloc[:num_choices]
                    .to_frame()
                    .to_csv(
        '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results'
        '/kaggle/working/individual_charts/data/' + title + '.csv',
        index=True
    )
    # compute max for chart scaling
    tmp_cmp = (response_counts.iloc[:num_choices] * 1.2).max()

# question we care about
question_of_interest = 'Are you currently a student? (high school, university, or graduate)'
# cache the two columns once to avoid repeated DataFrame.__getitem__ calls
country_col = responses_df_2022['In which country do you currently reside?']
student_col = responses_df_2022[question_of_interest]

for country, chart_title in [
    ('United States of America', 'Students status for Kaggle Survey participants from the USA'),
    ('India',                         'Students status for Kaggle Survey participants from India')
]:
    # build a boolean mask on the GPU
    mask = (country_col == country)
    # compute percentages in one pass (normalize=True) and stay on GPU
    percentages = (
        student_col[mask]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(1)
        .sort_index()
    )
    bar_chart_multiple_choice_24(
        response_counts=percentages,
        title=chart_title,
        y_axis_title='% of respondents',
        orientation=orientation_for_chart
    )


IndentationError: unexpected indent (<unknown>, line 10)

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_15_try_9.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_15_try_9.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[15], f)


In [ ]:
opt_output = Out.get(4)